# A.3: Testing and comparing

Round-trip tests, special tokens, and compression on held-out text compared with GPT-2.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.

# From A.1 The complete tokenizer (`bpe.py`)
import regex as re

from collections import Counter

# GPT-2's pre-tokenization pattern (Radford et al. 2019).
GPT2_PATTERN = r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

def merge(ids, pair, new_id):
    """Replace every occurrence of `pair` in the tuple `ids` with `new_id`."""
    out, i = [], 0
    while i < len(ids):
        if i + 1 < len(ids) and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            out.append(new_id)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return tuple(out)

class ByteBPE:
    def __init__(self, pattern=GPT2_PATTERN):
        self.pattern = re.compile(pattern)
        self.merges = {}                               # (id, id) -> new id
        self.vocab = {i: bytes([i]) for i in range(256)}  # id -> bytes
        self.special = {}                              # str -> id
        self.byte_ids = list(range(256))               # byte value -> id

    # ---- training: Steps 1 and 2 ---------------------------------------
    def train(self, text, vocab_size):
        chunks = Counter(self.pattern.findall(text))
        words = {tuple(c.encode("utf-8")): n for c, n in chunks.items()}
        for new_id in range(256, vocab_size):
            stats = Counter()
            for w, n in words.items():
                for pair in zip(w, w[1:]):
                    stats[pair] += n
            if not stats:
                break
            pair = max(stats, key=stats.get)   # ties: first pair seen
            words = {merge(w, pair, new_id): n for w, n in words.items()}
            self.merges[pair] = new_id
            self.vocab[new_id] = self.vocab[pair[0]] + self.vocab[pair[1]]

    # ---- encoding: Step 3 ----------------------------------------------
    def _encode_chunk(self, ids):
        while len(ids) >= 2:
            # Merge the pair that was learned earliest. Merge IDs grow with
            # rank, so the pair with the smallest merge ID has the lowest rank.
            pair = min(zip(ids, ids[1:]),
                       key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges:
                break
            ids = merge(ids, pair, self.merges[pair])
        return list(ids)

    def encode_ordinary(self, text):
        ids = []
        for chunk in self.pattern.findall(text):
            ids.extend(self._encode_chunk(
                tuple(self.byte_ids[b] for b in chunk.encode("utf-8"))))
        return ids

    def encode(self, text, allow_special=False):
        if not (allow_special and self.special):
            return self.encode_ordinary(text)
        split = "(" + "|".join(re.escape(s) for s in self.special) + ")"
        ids = []
        for part in re.split(split, text):
            if part in self.special:
                ids.append(self.special[part])
            elif part:
                ids.extend(self.encode_ordinary(part))
        return ids

    # ---- special tokens and decoding: Steps 4 and 5 --------------------
    def add_special_tokens(self, tokens):
        for t in tokens:
            if t not in self.special:
                new_id = len(self.vocab)
                self.special[t] = new_id
                self.vocab[new_id] = t.encode("utf-8")

    def decode(self, ids):
        data = b"".join(self.vocab[i] for i in ids)
        return data.decode("utf-8", errors="replace")

# From A.2 Training on tiny Shakespeare
text = open("shakespeare.txt", encoding="utf-8").read()

split = int(0.9 * len(text))

train, held_out = text[:split], text[split:]

tok = ByteBPE()

tok.train(train, vocab_size=512)

print([tok.vocab[i] for i in range(256, 276)])

print([tok.vocab[i] for i in range(500, 512)])


In [ ]:
tests = ["Hello, world!", "Emoji 🙂🚀 and accents: café, naïve, Zürich",
         "中文分词很难。日本語のテキスト。한국어 텍스트.",
         "def f(x):\n    return x ** 2  # square\n", "   spaces\t\ttabs\n\n\nnewlines   "]
print(all(tok.decode(tok.encode(t)) == t for t in tests))
print(len("中文分词很难。".encode("utf-8")), len(tok.encode("中文分词很难。")))


**Expected output**

Output:

```text
True
21 21
```


In [ ]:
tok.add_special_tokens(["<|endoftext|>"])
eot = tok.special["<|endoftext|>"]
s = "the end<|endoftext|>A new document"
print(eot, tok.encode(s, allow_special=True))
print(tok.encode(s).count(eot), tok.decode(tok.encode(s, allow_special=True)) == s)


**Expected output**

Output:

```text
512 [116, 257, 334, 267, 512, 65, 428, 119, 382, 99, 117, 109, 340]
0 True
```


In [ ]:
import tiktoken

n_bytes = len(held_out.encode("utf-8"))
tok_1024 = ByteBPE()
tok_1024.train(train, vocab_size=1024)
gpt2 = tiktoken.get_encoding("gpt2")
for name, n in [("ours, 512", len(tok.encode(held_out))),
                ("ours, 1024", len(tok_1024.encode(held_out))),
                ("GPT-2, 50257", len(gpt2.encode_ordinary(held_out)))]:
    print(name, n, round(n_bytes / n, 2))


**Expected output**

Output:

```text
ours, 512 59401 1.88
ours, 1024 49416 2.26
GPT-2, 50257 36059 3.09
```
